# Model Registry Audit

Inventory model artifacts and highlight missing or stale files.

Steps:
- Scan the models directory.
- Inspect expected model files and metrics.
- Summarize file sizes and formats.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from collections import Counter
from pathlib import Path

models_dir = REPO_ROOT / 'models'
summary = {
    'models_dir': str(models_dir),
    'counts': {},
    'by_ext': {},
    'largest': [],
    'expected': {},
}

if not models_dir.exists():
    print('Missing:', models_dir)
else:
    files = [p for p in models_dir.rglob('*') if p.is_file()]
    summary['counts']['total_files'] = len(files)
    ext_counts = Counter(p.suffix.lower() or 'no_ext' for p in files)
    summary['by_ext'] = dict(ext_counts.most_common(10))
    print('Total model files:', len(files))
    print('Top extensions:', summary['by_ext'])

    sizes = []
    for path in files:
        try:
            size = path.stat().st_size
        except OSError:
            size = 0
        sizes.append((size, path))
    sizes.sort(key=lambda item: item[0], reverse=True)
    summary['largest'] = [
        {
            'path': str(path.relative_to(REPO_ROOT)),
            'size_mb': round(size / 1024**2, 2),
        }
        for size, path in sizes[:8]
    ]
    print('Largest model files:')
    for item in summary['largest']:
        print(' -', item['path'], item['size_mb'], 'MB')


In [ ]:
# Check expected model artifacts.
expected = {
    'voice_emotion.pkl': REPO_ROOT / 'models' / 'voice_emotion.pkl',
    'voice_emotion_nn.pt': REPO_ROOT / 'models' / 'voice_emotion_nn.pt',
    'face_emotion_model': REPO_ROOT / 'models' / 'vision' / 'face_emotion' / 'model.pt',
    'face_emotion_metrics': REPO_ROOT / 'models' / 'vision' / 'face_emotion' / 'metrics.json',
    'fusion_meta_model': REPO_ROOT / 'models' / 'fusion' / 'fusion_meta_model.pkl',
}

for name, path in expected.items():
    summary['expected'][name] = {
        'exists': path.exists(),
        'path': str(path.relative_to(REPO_ROOT)) if path.exists() else str(path),
    }
    if path.exists():
        summary['expected'][name]['size_mb'] = round(path.stat().st_size / 1024**2, 2)
        print(name, '->', summary['expected'][name]['size_mb'], 'MB')
    else:
        print('Missing:', name, path)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_model_registry_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
